### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_medication
- _exponent._bronze_allscripts_tw_works.dbo_medication_de
- _exponent._bronze_allscripts_tw_works.dbo_item_medication (for PatientID linkage)

### Strategy:
- Join dbo_medication with dbo_medication_de via MedDictDE to get RxNorm codes
- Map RxNormCode/RxNormCodeNormalized to OMOP drug_concept_id
- Fallback: Map NDC to RxNorm via concept_relationship
- Calculate drug_exposure_end_date from start + DaysSupply
- Map route codes via domain_source_to_concept
- Use drug_type_concept_id based on context (prescription=32838, admin=32818)

### Notes:
- This notebook depends on source_to_person being populated
- provider_id and visit_occurrence_id will be NULL (mapping tables not yet created)
- RxNorm concept mapping requires OMOP vocabulary tables to be loaded
- dbo_medication does not have PatientID directly - linked via dbo_item_medication

# Transformation

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_allscripts.drug_exposure;

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_tw.drug_exposure;

In [0]:
%sql
DELETE FROM _exponent.omop_silver.drug_exposure
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_drug_exposure
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW standard_concept_drug AS
SELECT
    source_code,
    source_concept_id,
    drug_concept_id,
    drug_concept_name,
    drug_vocabulary_id,
    drug_concept_class_id
FROM (
    SELECT
        source_concept.concept_code AS source_code,
        source_concept.concept_id AS source_concept_id,
        COALESCE(
            direct_standard.concept_id,
            mapped_standard.concept_id
        ) AS drug_concept_id,
        COALESCE(
            direct_standard.concept_name,
            mapped_standard.concept_name
        ) AS drug_concept_name,
        COALESCE(
            direct_standard.vocabulary_id,
            mapped_standard.vocabulary_id
        ) AS drug_vocabulary_id,
        COALESCE(
            direct_standard.concept_class_id,
            mapped_standard.concept_class_id
        ) AS drug_concept_class_id,
        ROW_NUMBER() OVER (
            PARTITION BY source_concept.concept_code
            ORDER BY
                CASE source_concept.vocabulary_id
                    WHEN 'RxNorm' THEN 1
                    WHEN 'RxNorm Extension' THEN 2
                    ELSE 99
                END,
                CASE COALESCE(direct_standard.concept_class_id, mapped_standard.concept_class_id)
                    WHEN 'Clinical Drug' THEN 1
                    WHEN 'Branded Drug' THEN 2
                    WHEN 'Clinical Drug Comp' THEN 3
                    WHEN 'Branded Drug Comp' THEN 4
                    WHEN 'Ingredient' THEN 5
                    ELSE 99
                END,
                COALESCE(direct_standard.concept_id, mapped_standard.concept_id)
        ) AS rn
    FROM _exponent.omop.concept source_concept
    LEFT JOIN _exponent.omop.concept direct_standard
        ON source_concept.concept_id = direct_standard.concept_id
       AND direct_standard.domain_id = 'Drug'
       AND direct_standard.standard_concept = 'S'
       AND direct_standard.invalid_reason IS NULL
    LEFT JOIN _exponent.omop.concept_relationship concept_relationship
        ON source_concept.concept_id = concept_relationship.concept_id_1
       AND concept_relationship.relationship_id = 'Maps to'
    LEFT JOIN _exponent.omop.concept mapped_standard
        ON concept_relationship.concept_id_2 = mapped_standard.concept_id
       AND mapped_standard.domain_id = 'Drug'
       AND mapped_standard.standard_concept = 'S'
       AND mapped_standard.invalid_reason IS NULL
    WHERE source_concept.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
      AND source_concept.domain_id = 'Drug'
      AND source_concept.invalid_reason IS NULL
) ranked
WHERE rn = 1;

CREATE OR REPLACE TEMPORARY VIEW standard_concept_ndc AS
SELECT
    source_code,
    source_concept_id,
    drug_concept_id,
    drug_concept_name,
    drug_vocabulary_id,
    drug_concept_class_id
FROM (
    SELECT
        source_concept.concept_code AS source_code,
        source_concept.concept_id AS source_concept_id,
        COALESCE(
            direct_standard.concept_id,
            mapped_standard.concept_id
        ) AS drug_concept_id,
        COALESCE(
            direct_standard.concept_name,
            mapped_standard.concept_name
        ) AS drug_concept_name,
        COALESCE(
            direct_standard.vocabulary_id,
            mapped_standard.vocabulary_id
        ) AS drug_vocabulary_id,
        COALESCE(
            direct_standard.concept_class_id,
            mapped_standard.concept_class_id
        ) AS drug_concept_class_id,
        ROW_NUMBER() OVER (
            PARTITION BY source_concept.concept_code
            ORDER BY
                CASE COALESCE(direct_standard.concept_class_id, mapped_standard.concept_class_id)
                    WHEN 'Clinical Drug' THEN 1
                    WHEN 'Branded Drug' THEN 2
                    WHEN 'Clinical Drug Comp' THEN 3
                    WHEN 'Branded Drug Comp' THEN 4
                    WHEN 'Ingredient' THEN 5
                    ELSE 99
                END,
                COALESCE(direct_standard.concept_id, mapped_standard.concept_id)
        ) AS rn
    FROM _exponent.omop.concept source_concept
    LEFT JOIN _exponent.omop.concept direct_standard
        ON source_concept.concept_id = direct_standard.concept_id
       AND direct_standard.domain_id = 'Drug'
       AND direct_standard.standard_concept = 'S'
       AND direct_standard.invalid_reason IS NULL
    LEFT JOIN _exponent.omop.concept_relationship concept_relationship
        ON source_concept.concept_id = concept_relationship.concept_id_1
       AND concept_relationship.relationship_id = 'Maps to'
    LEFT JOIN _exponent.omop.concept mapped_standard
        ON concept_relationship.concept_id_2 = mapped_standard.concept_id
       AND mapped_standard.domain_id = 'Drug'
       AND mapped_standard.standard_concept = 'S'
       AND mapped_standard.invalid_reason IS NULL
    WHERE source_concept.vocabulary_id = 'NDC'
      AND source_concept.domain_id = 'Drug'
      AND source_concept.invalid_reason IS NULL
) ranked
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_drug_exposure AS
SELECT
  -- Drug concept mapping: prefer normalized RxNorm, then raw RxNorm, then NDC fallback
  COALESCE(standard_concept_drug.drug_concept_id, ndc_concept.drug_concept_id, 0) AS drug_concept_id,
  
  -- Start date
  DATE(med.StartFuzzySortAs) AS drug_exposure_start_date,
  med.StartFuzzySortAs AS drug_exposure_start_datetime,
  
  -- End date: use explicit end only when it is not before start; otherwise derive from days supply, then start.
  CASE
    WHEN med.EndFuzzySortAs IS NOT NULL
      AND DATE(med.EndFuzzySortAs) >= DATE(med.StartFuzzySortAs)
      THEN DATE(med.EndFuzzySortAs)
    WHEN COALESCE(med.DaysSupply, med.DaysToTake) IS NOT NULL
      AND COALESCE(med.DaysSupply, med.DaysToTake) > 0
      THEN DATE_ADD(DATE(med.StartFuzzySortAs), CAST(COALESCE(med.DaysSupply, med.DaysToTake) AS INT))
    ELSE DATE(med.StartFuzzySortAs)
  END AS drug_exposure_end_date,
  
  CASE
    WHEN med.EndFuzzySortAs IS NOT NULL
      AND DATE(med.EndFuzzySortAs) >= DATE(med.StartFuzzySortAs)
      THEN med.EndFuzzySortAs
    WHEN COALESCE(med.DaysSupply, med.DaysToTake) IS NOT NULL
      AND COALESCE(med.DaysSupply, med.DaysToTake) > 0
      THEN CAST(DATE_ADD(DATE(med.StartFuzzySortAs), CAST(COALESCE(med.DaysSupply, med.DaysToTake) AS INT)) AS TIMESTAMP)
    ELSE med.StartFuzzySortAs
  END AS drug_exposure_end_datetime,
  
  -- Verbatim end date (NULL out invalid dates < 1950)
  CASE 
    WHEN DATE(med.EndFuzzySortAs) >= '1950-01-01' THEN DATE(med.EndFuzzySortAs)
    ELSE NULL
  END AS verbatim_end_date,
  
  -- Drug type concept
  CASE
    WHEN med.Samples = 'Y' THEN 32839
    WHEN med.AdministeredByID IS NOT NULL THEN 32818
    ELSE 32838
  END AS drug_type_concept_id,
  
  NULL AS stop_reason,
  med.Refill AS refills,
  COALESCE(
    TRY_CAST(med.QuantityToDispense AS DOUBLE),
    TRY_CAST(med.SampleQuantity AS DOUBLE),
    TRY_CAST(med.AdministrationDoseQty AS DOUBLE)
  ) AS quantity,
  COALESCE(med.DaysSupply, med.DaysToTake) AS days_supply,
  COALESCE(med.FreeTextSIG, med.Instructions) AS sig,
  COALESCE(route_concept.omop_concept_id, 0) AS route_concept_id,
  COALESCE(med.AdministrationLot, med.SampleLot) AS lot_number,
  
  -- Source values
  COALESCE(
    NULLIF(TRIM(CAST(med_de.RxNormCodeNormalized AS STRING)), ''),
    NULLIF(TRIM(CAST(med_de.RxNormCode AS STRING)), ''),
    med.NDC,
    med_de.NDC,
    med_de.DrugName,
    med_de.DisplayName
  ) AS drug_source_value,
  -- Source concept ID: RxNorm source concept first, then NDC fallback
  COALESCE(standard_concept_drug.source_concept_id, ndc_concept.source_concept_id, 0) AS drug_source_concept_id,
  NULL AS route_source_value,
  COALESCE(med.Dose, med_de.UnitOfMeasure) AS dose_unit_source_value,
  
  -- FK source values
  CONCAT_WS(CHR(31), 'allscripts_tw', 'dbo_person', 'id', CAST(item_med.PatientID AS BIGINT)) AS person_source_value,
  
  CASE 
    WHEN COALESCE(med.PrescribedByID, med.AdministeredByID, med.SupervisedByID, med.WhoDidItID) IS NOT NULL 
    THEN CONCAT_WS(CHR(31), 'allscripts_tw', 'dbo_provider', 'id', CAST(COALESCE(med.PrescribedByID, med.AdministeredByID, med.SupervisedByID, med.WhoDidItID) AS BIGINT))
    ELSE NULL 
  END AS provider_source_value,
  
  CASE
    WHEN COALESCE(oah.VisitID, enc.VisitID) IS NOT NULL
    THEN CONCAT_WS(CHR(31), 'allscripts_tw', 'dbo_visit', 'id', CAST(COALESCE(oah.VisitID, enc.VisitID) AS BIGINT))
    ELSE NULL
  END AS visit_occurrence_source_value,
  
  NULL AS visit_detail_source_value,
  CONCAT_WS(CHR(31), 'allscripts_tw', 'dbo_medication', 'id', med.id) AS drug_exposure_source_value,
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works.dbo_medication med

LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works.dbo_medication_de med_de
  ON med.MedDictDE = med_de.ID

INNER JOIN _exponent._bronze_allscripts_tw_works.dbo_item_medication item_med
  ON med.ItemID = item_med.ID
  AND item_med.PatientID IS NOT NULL
  AND item_med.IsErrorFLAG = 'N'

LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_order_activity_header oah
  ON item_med.ActivityHeaderID = oah.ID

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter enc
  ON enc.ID = oah.EncounterID

LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_person
  ON dbo_person.id = item_med.PatientID

INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_tw', 'dbo_person', 'id', CAST(dbo_person.id AS BIGINT)) 
  AND stp.active_flag = TRUE

-- RxNorm code mapping (primary)
LEFT JOIN standard_concept_drug
  ON standard_concept_drug.source_code = COALESCE(
       NULLIF(TRIM(CAST(med_de.RxNormCodeNormalized AS STRING)), ''),
       NULLIF(TRIM(CAST(med_de.RxNormCode AS STRING)), '')
     )

-- NDC code mapping (fallback)
LEFT JOIN standard_concept_ndc ndc_concept
  ON ndc_concept.source_code = NULLIF(TRIM(COALESCE(CAST(med.NDC AS STRING), CAST(med_de.NDC AS STRING))), '')

LEFT JOIN _exponent.omop_mapping.domain_source_to_concept route_concept
  ON route_concept.source_id = CAST(COALESCE(med.RoutOfAdministrationDE, med.AdministrationRouteDE) AS INT)
  AND route_concept.domain_id = 'Route'
  AND route_concept.source_system = 'allscripts_tw'

WHERE med.StartFuzzySortAs IS NOT NULL
  AND DATE(med.StartFuzzySortAs) >= '1950-01-01'

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.drug_exposure AS t
USING (
  SELECT * FROM (
    SELECT 
      *,
      ROW_NUMBER() OVER (
        PARTITION BY drug_exposure_source_value, drug_type_concept_id 
        ORDER BY drug_exposure_start_date DESC
      ) AS rn
    FROM silver_drug_exposure
  ) WHERE rn = 1
) AS s
ON t.drug_exposure_source_value = s.drug_exposure_source_value 
   AND t.drug_type_concept_id = s.drug_type_concept_id

WHEN MATCHED AND (
     NOT (t.drug_concept_id <=> s.drug_concept_id)
  OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
  OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
  OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
  OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
  OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
  OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.refills <=> s.refills)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.days_supply <=> s.days_supply)
  OR NOT (t.sig <=> s.sig)
  OR NOT (t.route_concept_id <=> s.route_concept_id)
  OR NOT (t.lot_number <=> s.lot_number)
  OR NOT (t.drug_source_value <=> s.drug_source_value)
  OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
  OR NOT (t.route_source_value <=> s.route_source_value)
  OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.drug_concept_id             = s.drug_concept_id,
  t.drug_exposure_start_date    = s.drug_exposure_start_date,
  t.drug_exposure_start_datetime = s.drug_exposure_start_datetime,
  t.drug_exposure_end_date      = s.drug_exposure_end_date,
  t.drug_exposure_end_datetime  = s.drug_exposure_end_datetime,
  t.verbatim_end_date           = s.verbatim_end_date,
  t.drug_type_concept_id        = s.drug_type_concept_id,
  t.stop_reason                 = s.stop_reason,
  t.refills                     = s.refills,
  t.quantity                    = s.quantity,
  t.days_supply                 = s.days_supply,
  t.sig                         = s.sig,
  t.route_concept_id            = s.route_concept_id,
  t.lot_number                  = s.lot_number,
  t.drug_source_value           = s.drug_source_value,
  t.drug_source_concept_id      = s.drug_source_concept_id,
  t.route_source_value          = s.route_source_value,
  t.dose_unit_source_value      = s.dose_unit_source_value,
  t.person_source_value         = s.person_source_value,
  t.provider_source_value       = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value   = s.visit_detail_source_value,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  drug_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.drug_concept_id,
  s.drug_exposure_start_date,
  s.drug_exposure_start_datetime,
  s.drug_exposure_end_date,
  s.drug_exposure_end_datetime,
  s.verbatim_end_date,
  s.drug_type_concept_id,
  s.stop_reason,
  s.refills,
  s.quantity,
  s.days_supply,
  s.sig,
  s.route_concept_id,
  s.lot_number,
  s.drug_source_value,
  s.drug_source_concept_id,
  s.route_source_value,
  s.dose_unit_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.drug_exposure_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
%sql
-- Insert new mappings to source_to_drug_exposure
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
    source_system,
    drug_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.drug_exposure_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, drug_exposure_source_value, last_mod_tsp
    FROM _exponent.omop_silver.drug_exposure
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
  ON s.drug_exposure_source_value = x.drug_exposure_source_value;

In [0]:
%sql
MERGE INTO _exponent.omop_tw.drug_exposure AS gold
USING (
  SELECT * FROM (
    SELECT
      sde.drug_exposure_id,
      stp.person_id,
      s.drug_concept_id,
      s.drug_exposure_start_date,
      s.drug_exposure_start_datetime,
      s.drug_exposure_end_date,
      s.drug_exposure_end_datetime,
      s.verbatim_end_date,
      s.drug_type_concept_id,
      s.stop_reason,
      s.refills,
      s.quantity,
      s.days_supply,
      s.sig,
      s.route_concept_id,
      s.lot_number,
      NULL AS provider_id,
      COALESCE(stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
      NULL AS visit_detail_id,
      s.drug_source_value,
      s.drug_source_concept_id,
      s.route_source_value,
      s.dose_unit_source_value,
      ROW_NUMBER() OVER (
        PARTITION BY sde.drug_exposure_id 
        ORDER BY s.drug_exposure_start_date DESC
      ) AS rn
    FROM _exponent.omop_silver.drug_exposure s
    JOIN _exponent.omop_mapping.source_to_drug_exposure sde
      ON sde.drug_exposure_source_value = s.drug_exposure_source_value
      AND sde.source_system = 'allscripts_tw'
      AND sde.active_flag = TRUE
    JOIN _exponent.omop_mapping.source_to_person stp
      ON stp.person_source_value = s.person_source_value
      AND stp.source_system = 'allscripts_tw'
      AND stp.active_flag = TRUE
    LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
      ON stvo.visit_occurrence_source_value = s.visit_occurrence_source_value
      AND stvo.source_system = 'allscripts_tw'
      AND stvo.active_flag = TRUE
    LEFT JOIN _exponent.omop_tw.visit_occurrence vo
      ON vo.visit_source_value = s.visit_occurrence_source_value
    WHERE s.source_system = 'allscripts_tw'
  ) WHERE rn = 1
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED AND NOT (
  gold.person_id                    <=> src.person_id AND
  gold.drug_concept_id              <=> src.drug_concept_id AND
  gold.drug_exposure_start_date     <=> src.drug_exposure_start_date AND
  gold.drug_exposure_start_datetime <=> src.drug_exposure_start_datetime AND
  gold.drug_exposure_end_date       <=> src.drug_exposure_end_date AND
  gold.drug_exposure_end_datetime   <=> src.drug_exposure_end_datetime AND
  gold.verbatim_end_date            <=> src.verbatim_end_date AND
  gold.drug_type_concept_id         <=> src.drug_type_concept_id AND
  gold.stop_reason                  <=> src.stop_reason AND
  gold.refills                      <=> src.refills AND
  gold.quantity                     <=> src.quantity AND
  gold.days_supply                  <=> src.days_supply AND
  gold.sig                          <=> src.sig AND
  gold.route_concept_id             <=> src.route_concept_id AND
  gold.lot_number                   <=> src.lot_number AND
  gold.provider_id                  <=> src.provider_id AND
  gold.visit_occurrence_id          <=> src.visit_occurrence_id AND
  gold.visit_detail_id              <=> src.visit_detail_id AND
  gold.drug_source_value            <=> src.drug_source_value AND
  gold.drug_source_concept_id       <=> src.drug_source_concept_id AND
  gold.route_source_value           <=> src.route_source_value AND
  gold.dose_unit_source_value       <=> src.dose_unit_source_value
)

THEN UPDATE SET
  gold.person_id                    = src.person_id,
  gold.drug_concept_id              = src.drug_concept_id,
  gold.drug_exposure_start_date     = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date       = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime   = src.drug_exposure_end_datetime,
  gold.verbatim_end_date            = src.verbatim_end_date,
  gold.drug_type_concept_id         = src.drug_type_concept_id,
  gold.stop_reason                  = src.stop_reason,
  gold.refills                      = src.refills,
  gold.quantity                     = src.quantity,
  gold.days_supply                  = src.days_supply,
  gold.sig                          = src.sig,
  gold.route_concept_id             = src.route_concept_id,
  gold.lot_number                   = src.lot_number,
  gold.provider_id                  = src.provider_id,
  gold.visit_occurrence_id          = src.visit_occurrence_id,
  gold.visit_detail_id              = src.visit_detail_id,
  gold.drug_source_value            = src.drug_source_value,
  gold.drug_source_concept_id       = src.drug_source_concept_id,
  gold.route_source_value           = src.route_source_value,
  gold.dose_unit_source_value       = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.drug_exposure AS gold
USING (
  SELECT * FROM (
    SELECT
      sde.drug_exposure_id,
      stp.person_id,
      s.drug_concept_id,
      s.drug_exposure_start_date,
      s.drug_exposure_start_datetime,
      s.drug_exposure_end_date,
      s.drug_exposure_end_datetime,
      s.verbatim_end_date,
      s.drug_type_concept_id,
      s.stop_reason,
      s.refills,
      s.quantity,
      s.days_supply,
      s.sig,
      s.route_concept_id,
      s.lot_number,
      NULL AS provider_id,
      COALESCE(stvo.visit_occurrence_id, vo.visit_occurrence_id) AS visit_occurrence_id,
      NULL AS visit_detail_id,
      s.drug_source_value,
      s.drug_source_concept_id,
      s.route_source_value,
      s.dose_unit_source_value,
      ROW_NUMBER() OVER (
        PARTITION BY sde.drug_exposure_id 
        ORDER BY s.drug_exposure_start_date DESC
      ) AS rn
    FROM _exponent.omop_silver.drug_exposure s
    JOIN _exponent.omop_mapping.source_to_drug_exposure sde
      ON sde.drug_exposure_source_value = s.drug_exposure_source_value
      AND sde.source_system = 'allscripts_tw'
      AND sde.active_flag = TRUE
    JOIN _exponent.omop_mapping.source_to_person stp
      ON stp.person_source_value = s.person_source_value
      AND stp.source_system = 'allscripts_tw'
      AND stp.active_flag = TRUE
    LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
      ON stvo.visit_occurrence_source_value = s.visit_occurrence_source_value
      AND stvo.source_system = 'allscripts_tw'
      AND stvo.active_flag = TRUE
    LEFT JOIN _exponent.omop_tw.visit_occurrence vo
      ON vo.visit_source_value = s.visit_occurrence_source_value
    WHERE s.source_system = 'allscripts_tw'
  ) WHERE rn = 1
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED AND NOT (
  gold.person_id                    <=> src.person_id AND
  gold.drug_concept_id              <=> src.drug_concept_id AND
  gold.drug_exposure_start_date     <=> src.drug_exposure_start_date AND
  gold.drug_exposure_start_datetime <=> src.drug_exposure_start_datetime AND
  gold.drug_exposure_end_date       <=> src.drug_exposure_end_date AND
  gold.drug_exposure_end_datetime   <=> src.drug_exposure_end_datetime AND
  gold.verbatim_end_date            <=> src.verbatim_end_date AND
  gold.drug_type_concept_id         <=> src.drug_type_concept_id AND
  gold.stop_reason                  <=> src.stop_reason AND
  gold.refills                      <=> src.refills AND
  gold.quantity                     <=> src.quantity AND
  gold.days_supply                  <=> src.days_supply AND
  gold.sig                          <=> src.sig AND
  gold.route_concept_id             <=> src.route_concept_id AND
  gold.lot_number                   <=> src.lot_number AND
  gold.provider_id                  <=> src.provider_id AND
  gold.visit_occurrence_id          <=> src.visit_occurrence_id AND
  gold.visit_detail_id              <=> src.visit_detail_id AND
  gold.drug_source_value            <=> src.drug_source_value AND
  gold.drug_source_concept_id       <=> src.drug_source_concept_id AND
  gold.route_source_value           <=> src.route_source_value AND
  gold.dose_unit_source_value       <=> src.dose_unit_source_value
)

THEN UPDATE SET
  gold.person_id                    = src.person_id,
  gold.drug_concept_id              = src.drug_concept_id,
  gold.drug_exposure_start_date     = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date       = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime   = src.drug_exposure_end_datetime,
  gold.verbatim_end_date            = src.verbatim_end_date,
  gold.drug_type_concept_id         = src.drug_type_concept_id,
  gold.stop_reason                  = src.stop_reason,
  gold.refills                      = src.refills,
  gold.quantity                     = src.quantity,
  gold.days_supply                  = src.days_supply,
  gold.sig                          = src.sig,
  gold.route_concept_id             = src.route_concept_id,
  gold.lot_number                   = src.lot_number,
  gold.provider_id                  = src.provider_id,
  gold.visit_occurrence_id          = src.visit_occurrence_id,
  gold.visit_detail_id              = src.visit_detail_id,
  gold.drug_source_value            = src.drug_source_value,
  gold.drug_source_concept_id       = src.drug_source_concept_id,
  gold.route_source_value           = src.route_source_value,
  gold.dose_unit_source_value       = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

In [0]:
%sql
SELECT
  'drug_end_before_start' AS check_name,
  COUNT(*) AS fail_count
FROM _exponent.omop_tw.drug_exposure
WHERE drug_exposure_end_date < drug_exposure_start_date;

# Visit Linkage QA


In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN visit_occurrence_id IS NOT NULL THEN 1 ELSE 0 END) AS linked_rows,
  ROUND(100.0 * SUM(CASE WHEN visit_occurrence_id IS NOT NULL THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_linked
FROM _exponent.omop_tw.drug_exposure;
